In [ ]:
import json
from datetime import datetime

class AccountEntry:
    def __init__(self, date, description, amount):
        self.__date = self._validate_date(date)
        self.__description = description
        self.__amount = self._validate_amount(amount)

    def _validate_date(self, date):
        try:
            datetime.strptime(date, "%Y-%m-%d")
            return date
        except ValueError:
            raise ValueError("Invalid date format! Please enter YYYY-MM-DD (e.g., 2026-03-08)")

    def _validate_amount(self, amount):
        if not isinstance(amount, (int, float)) or amount <= 0:
            raise ValueError("Amount must be a positive number!")
        return amount

    def get_date(self):
        return self.__date

    def get_description(self):
        return self.__description

    def get_amount(self):
        return self.__amount

    def get_entry_type(self):
        raise NotImplementedError("Subclasses must implement this method!")

class IncomeEntry(AccountEntry):
    def __init__(self, date, description, amount, income_type):
        super().__init__(date, description, amount)
        self.__income_type = income_type

    def get_entry_type(self):
        return "income"

    def get_income_type(self):
        return self.__income_type

class ExpenseEntry(AccountEntry):
    def __init__(self, date, description, amount, expense_type):
        super().__init__(date, description, amount)
        self.__expense_type = expense_type

    def get_entry_type(self):
        return "expense"

    def get_expense_type(self):
        return self.__expense_type

class AccountingSystem:
    def __init__(self):
        self.__records = []

    def add_record(self, record):
        if isinstance(record, (IncomeEntry, ExpenseEntry)):
            self.__records.append(record)
            print(f" {record.get_entry_type()} record added successfully!")
        else:
            raise TypeError("Only IncomeEntry or ExpenseEntry type records can be added!")

    def calculate_total_income(self, filter_month=None):
        total = 0
        for record in self.__records:
            if record.get_entry_type() == "income":
                if filter_month:
                    record_month = record.get_date().split("-")[1]
                    if record_month != filter_month:
                        continue
                total += record.get_amount()
        return total

    def calculate_total_expense(self, filter_month=None):
        total = 0
        for record in self.__records:
            if record.get_entry_type() == "expense":
                if filter_month:
                    record_month = record.get_date().split("-")[1]
                    if record_month != filter_month:
                        continue
                total += record.get_amount()
        return total

    def calculate_balance(self, filter_month=None):
        income = self.calculate_total_income(filter_month)
        expense = self.calculate_total_expense(filter_month)
        return income - expense

    def calculate_by_category(self, entry_type, category):
        total = 0
        for record in self.__records:
            if record.get_entry_type() == entry_type:
                if entry_type == "income" and record.get_income_type() == category:
                    total += record.get_amount()
                elif entry_type == "expense" and record.get_expense_type() == category:
                    total += record.get_amount()
        return total

    def save_to_file(self, filename="accounting_data.json"):
        data = []
        for record in self.__records:
            record_data = {
                "date": record.get_date(),
                "description": record.get_description(),
                "amount": record.get_amount(),
                "type": record.get_entry_type()
            }
            if record.get_entry_type() == "income":
                record_data["category"] = record.get_income_type()
            else:
                record_data["category"] = record.get_expense_type()
            data.append(record_data)
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        print("💾 Data saved to file successfully!")

    def load_from_file(self, filename="accounting_data.json"):
        try:
            with open(filename, "r", encoding="utf-8") as f:
                data = json.load(f)
            self.__records = []
            for item in data:
                if item["type"] == "income":
                    record = IncomeEntry(
                        item["date"], item["description"], item["amount"], item["category"]
                    )
                else:
                    record = ExpenseEntry(
                        item["date"], item["description"], item["amount"], item["category"]
                    )
                self.__records.append(record)
            print("Data loaded from file successfully!")
        except FileNotFoundError:
            print("File not found! No data loaded.")

    def show_all_records(self):
        if not self.__records:
            print("No accounting records found!")
            return
        print("\n=================== All Accounting Records ===================")
        for idx, record in enumerate(self.__records, 1):
            entry_type = "Income" if record.get_entry_type() == "income" else "Expense"
            category = record.get_income_type() if record.get_entry_type() == "income" else record.get_expense_type()
            print(f"{idx}. [{entry_type}] Date: {record.get_date()} | Category: {category} | Description: {record.get_description()} | Amount: {record.get_amount()}")
        print("===================================================\n")

def main():
    system = AccountingSystem()
    print("🎉 Welcome to the OOP Accounting System 🎉")

    while True:
        print("\n===== Function Menu =====")
        print("1. Add Income Record")
        print("2. Add Expense Record")
        print("3. View All Records")
        print("4. View Income & Expense Summary")
        print("5. Statistics by Category")
        print("6. Statistics by Month")
        print("7. Save Data to File")
        print("8. Load Data from File")
        print("0. Exit System")
        print("====================")

        choice = input("Please enter the function number: ")

        if choice == "1":
            try:
                date = input("Please enter date (YYYY-MM-DD): ")
                desc = input("Please enter income description: ")
                amount = float(input("Please enter income amount: "))
                income_type = input("Please enter income category (Salary/Part-time/Investment/Others): ")
                record = IncomeEntry(date, desc, amount, income_type)
                system.add_record(record)
            except ValueError as e:
                print(f"Input Error: {e}")

        elif choice == "2":
            try:
                date = input("Please enter date (YYYY-MM-DD): ")
                desc = input("Please enter expense description: ")
                amount = float(input("Please enter expense amount: "))
                expense_type = input("Please enter expense category (Food/Transport/Entertainment/Others): ")
                record = ExpenseEntry(date, desc, amount, expense_type)
                system.add_record(record)
            except ValueError as e:
                print(f"Input Error: {e}")

        elif choice == "3":
            system.show_all_records()

        elif choice == "4":
            total_income = system.calculate_total_income()
            total_expense = system.calculate_total_expense()
            balance = system.calculate_balance()
            print("\n===== Income & Expense Summary =====")
            print(f"Total Income: {total_income}")
            print(f"Total Expense: {total_expense}")
            print(f"Current Balance: {balance}")
            print("====================")

        elif choice == "5":
            entry_type = input("Please enter statistics type (income/expense): ")
            if entry_type not in ["income", "expense"]:
                print("Invalid type! Please enter 'income' or 'expense'")
                continue
            category = input(f"Please enter {('income' if entry_type == 'income' else 'expense')} category: ")
            total = system.calculate_by_category(entry_type, category)
            print(f"\nTotal {('Income' if entry_type == 'income' else 'Expense')} for [{category}]: {total}")

        elif choice == "6":
            month = input("Please enter the month to statistics (two digits, e.g., 03): ")
            if len(month) != 2 or not month.isdigit():
                print("Invalid month format! Please enter two digits (01-12)")
                continue
            income = system.calculate_total_income(month)
            expense = system.calculate_total_expense(month)
            balance = income - expense
            print(f"\n===== {month} Monthly Statistics =====")
            print(f"Monthly Income: {income}")
            print(f"Monthly Expense: {expense}")
            print(f"Monthly Balance: {balance}")
            print("==========================")

        elif choice == "7":
            system.save_to_file()

        elif choice == "8":
            system.load_from_file()

        elif choice == "0":
            print("Thank you for using!")
            break

        else:
            print("Invalid number! Please try again.")

if __name__ == "__main__":
    main()